# COMP34212 -100 Coursework 

Notebook for the CIFAR-100 ResNet-18 experiments used in the report. It contains the data loading, model, training loop, grid search, repeated runs, summaries, and output download code.

The full run is long because it trains a baseline over three seeds, a 16-run grid search, and the tuned model over three seeds. On Colab, use a GPU runtime.

In [ ]:
# If running in Colab, this keeps the notebook self-contained.
!pip install -q torchvision pandas matplotlib

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
from __future__ import annotations

import csv
import itertools
import json
import math
import os
import random
import shutil
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms


## Dataset

CIFAR-100 is loaded through `torchvision.datasets.CIFAR100`. The official training split is divided into 40,000 training images and 10,000 validation images, while the official test split is kept for final evaluation.


In [ ]:
@dataclass
class DatasetBundle:
    train_loader: DataLoader
    val_loader: DataLoader
    test_loader: DataLoader
    class_names: list[str]


def create_cifar100_loaders(
    data_root: str | Path,
    batch_size: int,
    val_size: int = 10000,
    seed: int = 42,
    num_workers: int = 2,
) -> DatasetBundle:
    train_transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
    ])
    eval_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
    ])

    train_full_aug = datasets.CIFAR100(root=data_root, train=True, download=True, transform=train_transform)
    train_full_eval = datasets.CIFAR100(root=data_root, train=True, download=False, transform=eval_transform)
    test_dataset = datasets.CIFAR100(root=data_root, train=False, download=True, transform=eval_transform)

    rng = np.random.default_rng(seed)
    indices = np.arange(len(train_full_aug))
    rng.shuffle(indices)

    val_indices = indices[:val_size]
    train_indices = indices[val_size:]

    train_dataset = Subset(train_full_aug, train_indices.tolist())
    val_dataset = Subset(train_full_eval, val_indices.tolist())

    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        generator=generator,
    )
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    return DatasetBundle(train_loader, val_loader, test_loader, list(train_full_aug.classes))


## Model

The model is a CIFAR-adapted ResNet-18. The ImageNet-style large first convolution is replaced with a 3x3 stem for 32x32 images.

In [ ]:
def conv3x3(in_channels: int, out_channels: int, stride: int = 1) -> nn.Conv2d:
    return nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels: int, out_channels: int, stride: int = 1):
        super().__init__()
        self.conv1 = conv3x3(in_channels, out_channels, stride=stride)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(out_channels, out_channels)
        self.bn2 = nn.BatchNorm2d(out_channels)

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = self.shortcut(x)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x = self.relu(x + identity)
        return x


class CifarResNet(nn.Module):
    def __init__(self, layers: list[int], num_classes: int = 100, dropout: float = 0.0):
        super().__init__()
        self.in_channels = 64
        self.stem = nn.Sequential(conv3x3(3, 64), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.layer1 = self._make_layer(64, layers[0], stride=1)
        self.layer2 = self._make_layer(128, layers[1], stride=2)
        self.layer3 = self._make_layer(256, layers[2], stride=2)
        self.layer4 = self._make_layer(512, layers[3], stride=2)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.classifier = nn.Linear(512, num_classes)

        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.constant_(module.weight, 1)
                nn.init.constant_(module.bias, 0)

    def _make_layer(self, out_channels: int, blocks: int, stride: int) -> nn.Sequential:
        layers = [BasicBlock(self.in_channels, out_channels, stride=stride)]
        self.in_channels = out_channels
        for _ in range(1, blocks):
            layers.append(BasicBlock(self.in_channels, out_channels, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        return self.classifier(x)


def build_model(num_classes: int = 100, dropout: float = 0.0) -> nn.Module:
    return CifarResNet(layers=[2, 2, 2, 2], num_classes=num_classes, dropout=dropout)


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print('Trainable parameters:', count_parameters(build_model()))

## Training code

In [ ]:
@dataclass
class RunArtifacts:
    run_dir: Path
    metrics: dict[str, float]
    history: list[dict[str, float]]


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device() -> torch.device:
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def accuracy_topk(logits: torch.Tensor, targets: torch.Tensor, k: int = 1) -> float:
    _, predictions = torch.topk(logits, k=k, dim=1)
    return predictions.eq(targets.view(-1, 1)).any(dim=1).float().mean().item()


def run_epoch(model: nn.Module, loader: DataLoader, criterion: nn.Module, device: torch.device, optimizer=None) -> dict[str, float]:
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    total_top1 = 0.0
    total_top5 = 0.0
    total_samples = 0

    for inputs, targets in loader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            logits = model(inputs)
            loss = criterion(logits, targets)
            if training:
                loss.backward()
                optimizer.step()

        batch_size = inputs.size(0)
        total_loss += loss.item() * batch_size
        total_top1 += accuracy_topk(logits, targets, k=1) * batch_size
        total_top5 += accuracy_topk(logits, targets, k=min(5, logits.size(1))) * batch_size
        total_samples += batch_size

    return {
        'loss': total_loss / total_samples,
        'accuracy': total_top1 / total_samples,
        'top5_accuracy': total_top5 / total_samples,
    }


def save_history_csv(history: list[dict[str, float]], path: Path) -> None:
    with path.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(history[0].keys()))
        writer.writeheader()
        writer.writerows(history)


def plot_history(history: list[dict[str, float]], run_dir: Path) -> None:
    frame = pd.DataFrame(history)

    plt.figure(figsize=(7, 4))
    plt.plot(frame['epoch'], frame['train_accuracy'], label='Train')
    plt.plot(frame['epoch'], frame['val_accuracy'], label='Validation')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.tight_layout()
    plt.savefig(run_dir / 'accuracy.png', dpi=200)
    plt.close()

    plt.figure(figsize=(7, 4))
    plt.plot(frame['epoch'], frame['train_loss'], label='Train')
    plt.plot(frame['epoch'], frame['val_loss'], label='Validation')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.tight_layout()
    plt.savefig(run_dir / 'loss.png', dpi=200)
    plt.close()


def train_single(config: dict, tag: str | None = None) -> RunArtifacts:
    set_seed(int(config['seed']))
    output_root = Path(config.get('output_dir', 'outputs'))
    output_root.mkdir(parents=True, exist_ok=True)

    run_name = datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + config['name']
    if tag:
        run_name += '_' + tag
    run_dir = output_root / run_name
    run_dir.mkdir(parents=True, exist_ok=False)

    with (run_dir / 'config.json').open('w', encoding='utf-8') as handle:
        json.dump(config, handle, indent=2)

    device = get_device()
    print(f'Running on device: {device}')

    bundle = create_cifar100_loaders(
        data_root=config.get('data_root', 'data'),
        batch_size=int(config['batch_size']),
        val_size=int(config.get('val_size', 10000)),
        seed=int(config['seed']),
        num_workers=int(config.get('num_workers', 2)),
    )

    model = build_model(num_classes=100, dropout=float(config.get('dropout', 0.0))).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=float(config.get('label_smoothing', 0.0)))
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=float(config['learning_rate']),
        momentum=float(config.get('momentum', 0.9)),
        weight_decay=float(config.get('weight_decay', 0.0)),
    )

    scheduler = None
    if config.get('scheduler', 'none') == 'cosine':
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=int(config['epochs']),
            eta_min=float(config.get('min_learning_rate', 0.0005)),
        )

    history = []
    best_val_accuracy = -1.0
    epochs = int(config['epochs'])

    for epoch in range(1, epochs + 1):
        train_metrics = run_epoch(model, bundle.train_loader, criterion, device, optimizer=optimizer)
        val_metrics = run_epoch(model, bundle.val_loader, criterion, device)
        current_lr = optimizer.param_groups[0]['lr']
        if scheduler is not None:
            scheduler.step()

        record = {
            'epoch': epoch,
            'learning_rate': current_lr,
            'train_loss': train_metrics['loss'],
            'train_accuracy': train_metrics['accuracy'],
            'train_top5_accuracy': train_metrics['top5_accuracy'],
            'val_loss': val_metrics['loss'],
            'val_accuracy': val_metrics['accuracy'],
            'val_top5_accuracy': val_metrics['top5_accuracy'],
        }
        history.append(record)

        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"train_acc={train_metrics['accuracy']:.4f} | "
            f"val_acc={val_metrics['accuracy']:.4f} | "
            f"lr={current_lr:.5f}"
        )

        if val_metrics['accuracy'] > best_val_accuracy:
            best_val_accuracy = val_metrics['accuracy']
            torch.save(model.state_dict(), run_dir / 'best_model.pt')

    model.load_state_dict(torch.load(run_dir / 'best_model.pt', map_location=device))
    test_metrics = run_epoch(model, bundle.test_loader, criterion, device)

    metrics = {
        'best_val_accuracy': best_val_accuracy,
        'final_train_accuracy': history[-1]['train_accuracy'],
        'final_val_accuracy': history[-1]['val_accuracy'],
        'test_accuracy': test_metrics['accuracy'],
        'test_top5_accuracy': test_metrics['top5_accuracy'],
        'test_loss': test_metrics['loss'],
    }

    save_history_csv(history, run_dir / 'history.csv')
    plot_history(history, run_dir)
    with (run_dir / 'metrics.json').open('w', encoding='utf-8') as handle:
        json.dump(metrics, handle, indent=2)

    return RunArtifacts(run_dir=run_dir, metrics=metrics, history=history)

## Experiment settings

In [ ]:
BASELINE_CONFIG = {
    'name': 'cifar100_resnet18_baseline_25',
    'output_dir': 'outputs',
    'data_root': 'data',
    'seed': 42,
    'val_size': 10000,
    'num_workers': 2,
    'dropout': 0.0,
    'optimizer': 'sgd',
    'learning_rate': 0.01,
    'momentum': 0.9,
    'weight_decay': 0.0,
    'batch_size': 512,
    'epochs': 25,
    'label_smoothing': 0.0,
    'scheduler': 'none',
    'min_learning_rate': 0.0005,
}

GRID_BASE_CONFIG = {
    'name': 'cifar100_resnet18_grid',
    'output_dir': 'outputs',
    'data_root': 'data',
    'seed': 42,
    'val_size': 10000,
    'num_workers': 2,
    'dropout': 0.1,
    'optimizer': 'sgd',
    'learning_rate': 0.05,
    'momentum': 0.9,
    'weight_decay': 0.0005,
    'batch_size': 128,
    'epochs': 12,
    'label_smoothing': 0.0,
    'scheduler': 'cosine',
    'min_learning_rate': 0.0005,
}

TUNED_CONFIG = {
    'name': 'cifar100_resnet18_tuned_25',
    'output_dir': 'outputs',
    'data_root': 'data',
    'seed': 42,
    'val_size': 10000,
    'num_workers': 2,
    'dropout': 0.0,
    'optimizer': 'sgd',
    'learning_rate': 0.1,
    'momentum': 0.9,
    'weight_decay': 0.0005,
    'batch_size': 64,
    'epochs': 25,
    'label_smoothing': 0.0,
    'scheduler': 'cosine',
    'min_learning_rate': 0.0005,
}

GRID = {
    'learning_rate': [0.1, 0.05],
    'batch_size': [64, 128],
    'weight_decay': [0.0005, 0.0001],
    'dropout': [0.0, 0.2],
}

SEEDS = [42, 43, 44]

## Run experiments

The switches below let the same notebook run the full set of experiments or only selected parts. The full reproduction run is 22 trainings: three baseline runs, 16 grid-search runs, and three tuned runs.

In [ ]:
RUN_BASELINE_REPEATS = True
RUN_GRID_SEARCH = True
RUN_TUNED_REPEATS = True

In [ ]:
def run_repeated(config: dict, seeds: list[int], label: str) -> pd.DataFrame:
    rows = []
    for seed in seeds:
        run_config = dict(config)
        run_config['seed'] = int(seed)
        print('\n' + '=' * 80)
        print(f'{label} seed {seed}')
        print('=' * 80)
        artifacts = train_single(run_config, tag=f'seed-{seed}')
        rows.append({'seed': int(seed), 'run_dir': str(artifacts.run_dir), **artifacts.metrics})

    frame = pd.DataFrame(rows).sort_values('test_accuracy', ascending=False)
    outputs = Path(config.get('output_dir', 'outputs'))
    frame.to_csv(outputs / f'{label}_repeat_summary.csv', index=False)

    stats = {'n_runs': len(frame)}
    for column in ['best_val_accuracy', 'final_train_accuracy', 'final_val_accuracy', 'test_accuracy', 'test_top5_accuracy', 'test_loss']:
        stats[f'{column}_mean'] = float(frame[column].mean())
        stats[f'{column}_std'] = float(frame[column].std(ddof=1)) if len(frame) > 1 else 0.0

    with (outputs / f'{label}_repeat_summary_stats.json').open('w', encoding='utf-8') as handle:
        json.dump(stats, handle, indent=2)

    print(stats)
    return frame


def run_grid_search(base_config: dict, grid: dict[str, list]) -> pd.DataFrame:
    keys = list(grid.keys())
    rows = []
    combinations = list(itertools.product(*[grid[key] for key in keys]))

    for index, values in enumerate(combinations, start=1):
        overrides = dict(zip(keys, values))
        run_config = dict(base_config)
        run_config.update(overrides)
        tag = '_'.join(f'{key}-{value}' for key, value in overrides.items())

        print('\n' + '=' * 80)
        print(f'Grid trial {index}/{len(combinations)}: {overrides}')
        print('=' * 80)
        artifacts = train_single(run_config, tag=tag)
        rows.append({**overrides, 'run_dir': str(artifacts.run_dir), **artifacts.metrics})

    frame = pd.DataFrame(rows).sort_values('best_val_accuracy', ascending=False)
    outputs = Path(base_config.get('output_dir', 'outputs'))
    frame.to_csv(outputs / 'search_summary.csv', index=False)

    best = frame.iloc[0].to_dict()
    with (outputs / 'best_grid_result.json').open('w', encoding='utf-8') as handle:
        json.dump(best, handle, indent=2)

    print('Best grid result:')
    print(frame.head(1).T)
    return frame

In [ ]:
outputs = Path('outputs')
outputs.mkdir(exist_ok=True)

baseline_summary = None
grid_summary = None
tuned_summary = None

if RUN_BASELINE_REPEATS:
    baseline_summary = run_repeated(BASELINE_CONFIG, SEEDS, 'baseline_25')

if RUN_GRID_SEARCH:
    grid_summary = run_grid_search(GRID_BASE_CONFIG, GRID)

if RUN_TUNED_REPEATS:
    tuned_summary = run_repeated(TUNED_CONFIG, SEEDS, 'tuned_25')

## Summaries and figures

In [ ]:
def load_summary(name: str) -> pd.DataFrame | None:
    path = Path('outputs') / name
    if path.exists():
        return pd.read_csv(path)
    return None

baseline_summary = baseline_summary if baseline_summary is not None else load_summary('baseline_25_repeat_summary.csv')
tuned_summary = tuned_summary if tuned_summary is not None else load_summary('tuned_25_repeat_summary.csv')
grid_summary = grid_summary if grid_summary is not None else load_summary('search_summary.csv')

if baseline_summary is not None and tuned_summary is not None:
    comparison = pd.DataFrame([
        {
            'setting': 'baseline_25',
            'test_acc_mean': baseline_summary['test_accuracy'].mean() * 100,
            'test_acc_std': baseline_summary['test_accuracy'].std(ddof=1) * 100,
            'val_acc_mean': baseline_summary['best_val_accuracy'].mean() * 100,
            'top5_mean': baseline_summary['test_top5_accuracy'].mean() * 100,
            'loss_mean': baseline_summary['test_loss'].mean(),
        },
        {
            'setting': 'tuned_25',
            'test_acc_mean': tuned_summary['test_accuracy'].mean() * 100,
            'test_acc_std': tuned_summary['test_accuracy'].std(ddof=1) * 100,
            'val_acc_mean': tuned_summary['best_val_accuracy'].mean() * 100,
            'top5_mean': tuned_summary['test_top5_accuracy'].mean() * 100,
            'loss_mean': tuned_summary['test_loss'].mean(),
        },
    ])
    comparison.to_csv('outputs/baseline_vs_tuned_25_comparison.csv', index=False)
    display(comparison)

if grid_summary is not None:
    display(grid_summary.head(16))

In [ ]:
def history_path(run_dir: str) -> Path:
    return Path(run_dir) / 'history.csv'


def load_histories(summary: pd.DataFrame) -> list[pd.DataFrame]:
    histories = []
    for _, row in summary.sort_values('seed').iterrows():
        path = history_path(row['run_dir'])
        if path.exists():
            history = pd.read_csv(path)
            history['seed'] = int(row['seed'])
            histories.append(history)
    return histories


def plot_baseline_vs_tuned(baseline_summary: pd.DataFrame, tuned_summary: pd.DataFrame) -> None:
    baseline_histories = load_histories(baseline_summary)
    tuned_histories = load_histories(tuned_summary)
    if not baseline_histories or not tuned_histories:
        print('Missing history files; skipping learning curve plot.')
        return

    def mean_std(histories, column):
        epochs = histories[0]['epoch'].to_numpy()
        values = np.vstack([h[column].to_numpy() for h in histories]) * 100
        return epochs, values.mean(axis=0), values.std(axis=0, ddof=1)

    b_ep, b_mean, b_std = mean_std(baseline_histories, 'val_accuracy')
    t_ep, t_mean, t_std = mean_std(tuned_histories, 'val_accuracy')
    b_test = baseline_summary['test_accuracy'].to_numpy() * 100
    t_test = tuned_summary['test_accuracy'].to_numpy() * 100

    fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.25), gridspec_kw={'width_ratios': [1.45, 1]})
    ax = axes[0]
    ax.plot(b_ep, b_mean, label='Baseline', color='#4e6678', linewidth=2)
    ax.fill_between(b_ep, b_mean - b_std, b_mean + b_std, color='#4e6678', alpha=0.15, linewidth=0)
    ax.plot(t_ep, t_mean, label='Tuned', color='#c94f44', linewidth=2)
    ax.fill_between(t_ep, t_mean - t_std, t_mean + t_std, color='#c94f44', alpha=0.15, linewidth=0)
    ax.set_title('(a) Validation accuracy over training')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Validation accuracy (%)')
    ax.set_ylim(0, 80)
    ax.grid(True, alpha=0.22)
    ax.legend(frameon=False, loc='lower right')

    ax = axes[1]
    means = [b_test.mean(), t_test.mean()]
    stds = [b_test.std(ddof=1), t_test.std(ddof=1)]
    x = np.arange(2)
    ax.bar(x, means, yerr=stds, capsize=4, color=['#4e6678', '#c94f44'], width=0.46)
    ax.set_title('(b) Final test accuracy')
    ax.set_ylabel('Test accuracy (%)')
    ax.set_xticks(x, ['Baseline', 'Tuned'])
    ax.set_ylim(0, 100)
    ax.grid(axis='y', alpha=0.22)
    for xi, value in zip(x, means):
        ax.text(xi, value + 2, f'{value:.1f}%', ha='center', fontsize=8)

    fig.suptitle('Baseline versus tuned ResNet-18', fontweight='bold', y=1.02)
    fig.tight_layout()
    fig.savefig('outputs/figure1_learning_curves.png', dpi=300, bbox_inches='tight')
    plt.show()


def plot_grid_heatmaps(grid_summary: pd.DataFrame) -> None:
    if grid_summary is None:
        return
    frame = grid_summary.copy()
    frame['test_accuracy'] = frame['test_accuracy'] * 100

    fig, axes = plt.subplots(2, 2, figsize=(7.0, 5.0), constrained_layout=True)
    settings = [(0.0005, 0.0), (0.0005, 0.2), (0.0001, 0.0), (0.0001, 0.2)]
    vmin = frame['test_accuracy'].min()
    vmax = frame['test_accuracy'].max()

    for ax, (weight_decay, dropout) in zip(axes.ravel(), settings):
        sub = frame[(frame['weight_decay'] == weight_decay) & (frame['dropout'] == dropout)]
        pivot = sub.pivot(index='learning_rate', columns='batch_size', values='test_accuracy').sort_index(ascending=False)
        image = ax.imshow(pivot.values, vmin=vmin, vmax=vmax, cmap='YlGnBu')
        ax.set_title(f'WD={weight_decay:g}, dropout={dropout:g}', fontsize=9)
        ax.set_xticks(range(len(pivot.columns)), [str(c) for c in pivot.columns])
        ax.set_yticks(range(len(pivot.index)), [str(i) for i in pivot.index])
        ax.set_xlabel('Batch size')
        ax.set_ylabel('Learning rate')
        for i in range(pivot.shape[0]):
            for j in range(pivot.shape[1]):
                ax.text(j, i, f'{pivot.values[i, j]:.1f}%', ha='center', va='center', fontsize=8, fontweight='bold')

    fig.colorbar(image, ax=axes.ravel().tolist(), shrink=0.82, label='Test accuracy (%)')
    fig.suptitle('Extended 16-run factorial grid search on CIFAR-100', fontweight='bold')
    fig.savefig('outputs/figure2_grid_heatmaps.png', dpi=300, bbox_inches='tight')
    plt.show()

if baseline_summary is not None and tuned_summary is not None:
    plot_baseline_vs_tuned(baseline_summary, tuned_summary)

if grid_summary is not None:
    plot_grid_heatmaps(grid_summary)

## Download outputs from Colab

In [ ]:
if 'google.colab' in str(get_ipython()):
    from google.colab import files
    zip_path = shutil.make_archive('/content/cifar100_coursework_outputs', 'zip', 'outputs')
    files.download(zip_path)
else:
    print('Outputs are in:', Path('outputs').resolve())